# IOAI — 2026 Local Onia Museums (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/muzee.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-museums.zip', '/tmp/muzee.zip')
    with zipfile.ZipFile('/tmp/muzee.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/muzee.csv' if os.path.exists('data/muzee.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Museums (Muzee) — 박물관 데이터 분석 (RoAI/ONIA 2026 Local)

루마니아 박물관 등록부(`muzee.csv`)에 대한 **순수 데이터 분석** 과제(8 서브태스크): 행/열 수, 특정 군(județ)의 개수, 연도별 집계, 최빈값 비율 등 pandas 집계로 답을 구한다. ML/라벨 없음.

**제출**: `submission.csv` 롱포맷 `id, subtaskID, answer`.
**채점(로컬)**: 답이 입력의 결정적 함수이므로 grader 가 `muzee.csv` 에서 레퍼런스를 재계산해 exact match.
**데이터**: `data/muzee.csv`. 원본: platform.olimpiada-ai.ro/competitions/12

> **베이스라인(부분 풀이)**: 쉬운 집계 서브태스크(1·2·3·5)만 계산하고 4·6·7·8 은 TODO(임시값).
> 실행하면 `data/submission.csv` 가 생성되어 Submissions 에 잡히고, 서브태스크 균등평균 acc≈0.5 가 나옵니다.
> 나머지를 직접 채우면 1.0 에 가까워집니다. 모범답안은 Solution/ 참고.

In [ ]:
import pandas as pd, numpy as np

root_path = "data"
muzee = pd.read_csv(f"{root_path}/muzee.csv")
muzee.head()

In [ ]:
# === 쉬운 서브태스크 (직접 계산) ===
subtask1 = len(muzee["denumirea (română)"])                  # 박물관 수(행 수)
subtask2 = len(muzee[muzee["județul"] == "București"])       # București 군 박물관 수
subtask3 = int((muzee.isna().sum() > 0).sum())               # 결측이 있는 컬럼 수
subtask5 = (muzee.groupby("județul")["codul entității muzeale"]
                 .nunique().sort_values(ascending=False))     # 군별 고유 박물관 수
print(subtask1, subtask2, subtask3, len(subtask5))

In [ ]:
# === TODO: 아래 서브태스크를 직접 구현하세요 (지금은 임시값 0) ===
# subtask4: 가장 많은 박물관이 설립된 연도 ('anul înființării' 파싱 필요)
# subtask6: 박물관별 데이터 완성도(%) = 채워진 칸 수 / 전체 컬럼 수 * 100, 소수 2자리
# subtask7: subtask6 의 평균
# subtask8: 완성도가 최댓값인 박물관 비율(%)
subtask4 = 0
subtask6 = pd.Series(0, index=muzee.index)
subtask7 = 0
subtask8 = 0

In [ ]:
# === 제출(롱포맷): id, subtaskID, answer ===
def build_subtask(id, sid, ans):
    return pd.DataFrame({"id": id, "subtaskID": sid, "answer": ans})
subtasks = [
    ([1], 1, subtask1), ([2], 2, subtask2), ([3], 3, subtask3), ([4], 4, subtask4),
    (subtask5.index, 5, subtask5.values),
    (muzee["_id"], 6, subtask6), ([7], 7, subtask7), ([8], 8, subtask8),
]
submission = pd.concat([build_subtask(i, s, a) for i, s, a in subtasks])
submission.to_csv(f"{root_path}/submission.csv", index=False)
submission.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)